# Household-aware any-wasting prediction with XGBoost and LightGBM

This notebook compares four strategies: **XGBoost + SMOTENC (without child weights)**, **LightGBM + SMOTENC (without child weights)**, **XGBoost + child sample weights (without SMOTENC)**, and **LightGBM + child sample weights (without SMOTENC)**.

**Sampling-unit correction:** children from the same household are not independent. A household group ID is created from `division` + `hh1`. Both the held-out split and cross-validation use `StratifiedGroupKFold`, so no household can appear in both training and validation/test data.

**Missing data:** MICE is performed separately inside each training fold using `miceforest`. The fitted imputation kernel is then applied to that fold's validation data. This prevents validation/test information from leaking into imputation.

**Encoding order:** categorical predictors are imputed as categorical variables, SMOTENC is applied to the imputed training data where requested, and one-hot encoding is fitted/applied only afterwards.

No model results are claimed until the notebook is run against the revised MICS dataset.

## 1. Install dependencies

In [ ]:
import importlib.util, subprocess, sys
packages = {
    "numpy":"numpy", "pandas":"pandas", "sklearn":"scikit-learn",
    "imblearn":"imbalanced-learn", "xgboost":"xgboost",
    "lightgbm":"lightgbm", "miceforest":"miceforest",
    "shap":"shap", "matplotlib":"matplotlib"
}
missing = [pip for mod, pip in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Dependencies ready.")

## 2. Imports and reproducibility

In [ ]:
import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import miceforest as mf
import shap

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTENC
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay, precision_recall_curve
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
SEED = 42
CV_FOLDS = 5
np.random.seed(SEED)

## 3. Load the revised MICS child dataset

In [ ]:
DATA_PATH = Path("MICS-child-ISRC-revised.dta")
if not DATA_PATH.exists():
    candidates = [p for p in [Path("MICS-child-ISRC.dta"), *Path(".").glob("*.dta")] if p.exists()]
    if len(candidates) != 1:
        raise FileNotFoundError(f"Set DATA_PATH explicitly; found: {candidates}")
    DATA_PATH = candidates[0]

df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print("Loaded:", DATA_PATH, "| shape:", df.shape)
print("Weight-like fields:", [c for c in df.columns if "weight" in c.lower()])

## 4. Define outcome, predictors, weights and household groups

The household identifier is formed by combining the division code with `hh1`. This avoids treating the same household number in different divisions as the same household. The identifier is used only for splitting, never as a predictor.

Rows are excluded only if the outcome, positive child weight, or household grouping keys are unavailable. Missing predictor values are retained for MICE.

In [ ]:
TARGET = "any_wasting"
PREDICTORS = [
    "age_m","female","division","urban","wb4","parity_cat","diarrhoea_2w",
    "cough_2w","breathing_2w","fuel_cat","water_cat","toilet_cat","edu_level",
    "electricity","wealth","birth_order","hh_size","water_treated",
    "water_boil_filter","handwash_place","handwash_water","floor_quality","ari_chest"
]
CATEGORICAL_COLUMNS = [
    "female","division","urban","parity_cat","diarrhoea_2w","cough_2w",
    "breathing_2w","fuel_cat","water_cat","toilet_cat","edu_level","electricity",
    "wealth","birth_order","water_treated","water_boil_filter","handwash_place",
    "handwash_water","floor_quality","ari_chest"
]
NUMERIC_COLUMNS = [c for c in PREDICTORS if c not in CATEGORICAL_COLUMNS]
WEIGHT_CANDIDATES = ["ch_weight", "chweight"]
WEIGHT_COL = next((c for c in WEIGHT_CANDIDATES if c in df.columns), None)

required = [TARGET] + PREDICTORS + ["hh1", "division"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")
if WEIGHT_COL is None:
    raise KeyError("Child sample weight not found: expected ch_weight or chweight.")

model_df = df[required + [WEIGHT_COL]].copy()
model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="coerce")
model_df[WEIGHT_COL] = pd.to_numeric(model_df[WEIGHT_COL], errors="coerce")
model_df = model_df.dropna(subset=[TARGET, WEIGHT_COL, "hh1", "division"])
model_df = model_df.loc[model_df[WEIGHT_COL] > 0].copy()
model_df[TARGET] = model_df[TARGET].astype(int)
if sorted(model_df[TARGET].unique()) != [0, 1]:
    raise ValueError("any_wasting must be coded 0/1.")

# Stable composite group key; hh1 may repeat across divisions.
model_df["household_group"] = (
    model_df["division"].astype(str).str.strip() + "__" +
    model_df["hh1"].astype(str).str.strip()
)
X = model_df[PREDICTORS].copy()
y = model_df[TARGET].copy()
w = model_df[WEIGHT_COL].astype(float).copy()
groups = model_df["household_group"].copy()

print("Weight column:", WEIGHT_COL)
print("Rows retained:", len(model_df))
print("Unique households:", groups.nunique())
print("Children per household (median/max):",
      model_df.groupby("household_group").size().median(),
      model_df.groupby("household_group").size().max())
print("Outcome counts:", y.value_counts().sort_index().to_dict())
print("Missing predictor cells:", int(X.isna().sum().sum()))

## 5. Household-aware held-out split

The first fold of a shuffled `StratifiedGroupKFold` is reserved as the test set. The remaining folds form the training set. This preserves approximate outcome balance while ensuring that household groups are disjoint. Because groups are indivisible, the test proportion may not be exactly 20%.

In [ ]:
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(outer_cv.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
w_train, w_test = w.iloc[train_idx].copy(), w.iloc[test_idx].copy()
g_train, g_test = groups.iloc[train_idx].copy(), groups.iloc[test_idx].copy()

assert set(g_train).isdisjoint(set(g_test)), "Household leakage detected!"
print(f"Train rows: {len(X_train):,} | households: {g_train.nunique():,}")
print(f"Test rows:  {len(X_test):,} | households: {g_test.nunique():,}")
print("Actual test fraction:", round(len(X_test) / len(X), 4))
print("Train prevalence:", round(y_train.mean(), 4), "| test prevalence:", round(y_test.mean(), 4))

## 6. MICE imputation, SMOTENC and post-resampling one-hot encoding

Each fold gets a fresh MICE kernel fitted on that fold's training observations only. `miceforest` uses chained conditional models and supports categorical predictors. Categorical columns are explicitly cast to pandas category dtype before fitting.

For the SMOTENC strategies, categorical columns remain categorical during resampling. One-hot encoding is deliberately delayed until after SMOTENC. For the weighted/no-SMOTE strategies, the same MICE and one-hot steps are used without resampling.

The child survey weight is not a class-balancing weight. In the SMOTENC strategies it is not passed to model fitting.

In [ ]:
# Define categorical domains once from predictor values only (no outcome/weight information).
# This ensures train/validation/test use identical category levels.
CATEGORY_LEVELS = {
    c: list(pd.Series(X[c].dropna().unique()).sort_values(key=lambda s: s.astype(str)))
    for c in CATEGORICAL_COLUMNS
}

def cast_for_mice(frame):
    out = frame[PREDICTORS].copy()
    for col in CATEGORICAL_COLUMNS:
        out[col] = pd.Categorical(out[col], categories=CATEGORY_LEVELS[col])
    for col in NUMERIC_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype(float)
    return out

def mice_impute_train_valid(Xf, Xv, iterations=5):
    train_mice = cast_for_mice(Xf)
    valid_mice = cast_for_mice(Xv)
    kernel = mf.ImputationKernel(
        train_mice,
        num_datasets=1,
        save_all_iterations_data=True,
        random_state=SEED
    )
    kernel.mice(iterations=iterations, verbose=False)
    train_done = kernel.complete_data(dataset=0)
    valid_kernel = kernel.impute_new_data(
        new_data=valid_mice,
        datasets=0,
        iterations=iterations,
        verbose=False
    )
    valid_done = valid_kernel.complete_data(dataset=0)

    # Restore stable categorical dtype for SMOTENC and the encoder.
    for col in CATEGORICAL_COLUMNS:
        train_done[col] = pd.Categorical(train_done[col], categories=CATEGORY_LEVELS[col])
        valid_done[col] = pd.Categorical(valid_done[col], categories=CATEGORY_LEVELS[col])
    for col in NUMERIC_COLUMNS:
        train_done[col] = pd.to_numeric(train_done[col], errors="coerce").astype(float)
        valid_done[col] = pd.to_numeric(valid_done[col], errors="coerce").astype(float)

    if train_done.isna().any().any() or valid_done.isna().any().any():
        raise ValueError("MICE left missing values; inspect variables with excessive missingness.")
    return train_done, valid_done, kernel

def post_smote_onehot(Xfit, Xvalid, yfit, use_smote):
    if use_smote:
        cat_idx = [Xfit.columns.get_loc(c) for c in CATEGORICAL_COLUMNS]
        counts = pd.Series(yfit).value_counts()
        if counts.min() < 2:
            raise ValueError("Too few minority observations in this training fold for SMOTENC.")
        sampler = SMOTENC(
            categorical_features=cat_idx,
            random_state=SEED,
            k_neighbors=min(5, int(counts.min()) - 1)
        )
        X_res, y_res = sampler.fit_resample(Xfit, yfit)
        if not isinstance(X_res, pd.DataFrame):
            X_res = pd.DataFrame(X_res, columns=Xfit.columns)
        for col in CATEGORICAL_COLUMNS:
            X_res[col] = pd.Categorical(X_res[col], categories=CATEGORY_LEVELS[col])
        Xfit = X_res
        yfit = np.asarray(y_res, dtype=int)

    encoder = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),
            ("numeric", "passthrough", NUMERIC_COLUMNS)
        ],
        remainder="drop"
    )
    Xfit_enc = encoder.fit_transform(Xfit)
    Xvalid_enc = encoder.transform(Xvalid)
    feature_names = encoder.get_feature_names_out()
    return Xfit_enc, Xvalid_enc, np.asarray(yfit, dtype=int), encoder, feature_names

MODEL_PARAMS = {
    "XGBoost": {
        "n_estimators": 700, "max_depth": 5, "learning_rate": 0.05,
        "min_child_weight": 3, "subsample": 0.85, "colsample_bytree": 0.85,
        "reg_lambda": 5.0
    },
    "LightGBM": {
        "n_estimators": 700, "num_leaves": 31, "max_depth": -1,
        "learning_rate": 0.05, "min_child_samples": 25,
        "subsample": 0.85, "subsample_freq": 1,
        "colsample_bytree": 0.85, "reg_lambda": 5.0
    }
}

def make_estimator(model_name):
    if model_name == "XGBoost":
        return XGBClassifier(
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=-1, random_state=SEED,
            verbosity=0, **MODEL_PARAMS[model_name]
        )
    return LGBMClassifier(
        objective="binary", n_jobs=-1, random_state=SEED,
        verbosity=-1, **MODEL_PARAMS[model_name]
    )

def fit_configuration(model_name, strategy, Xf, yf, wf, Xv):
    use_smote = strategy == "SMOTENC_no_ch_weight"
    use_weights = strategy == "ch_weight_no_SMOTENC"
    Xf_imp, Xv_imp, mice_kernel = mice_impute_train_valid(Xf, Xv)
    Xf_enc, Xv_enc, yfit, encoder, feature_names = post_smote_onehot(
        Xf_imp, Xv_imp, yf, use_smote
    )
    estimator = make_estimator(model_name)
    if use_weights:
        fit_weights = np.asarray(wf, dtype=float)
        fit_weights = fit_weights / fit_weights.mean()
        estimator.fit(Xf_enc, yfit, sample_weight=fit_weights)
    else:
        estimator.fit(Xf_enc, yfit)
    return {
        "model": estimator, "encoder": encoder, "mice_kernel": mice_kernel,
        "feature_names": feature_names, "X_train_imputed": Xf_imp,
        "X_valid_imputed": Xv_imp, "X_train_encoded": Xf_enc,
        "X_valid_encoded": Xv_enc
    }

def predict_bundle(bundle, Xnew):
    # Impute new observations using the fitted training kernel.
    new_mice = cast_for_mice(Xnew)
    new_kernel = bundle["mice_kernel"].impute_new_data(
        new_data=new_mice, datasets=0, iterations=5, verbose=False
    )
    new_done = new_kernel.complete_data(dataset=0)
    for col in CATEGORICAL_COLUMNS:
        new_done[col] = pd.Categorical(new_done[col], categories=CATEGORY_LEVELS[col])
    for col in NUMERIC_COLUMNS:
        new_done[col] = pd.to_numeric(new_done[col], errors="coerce").astype(float)
    encoded = bundle["encoder"].transform(new_done)
    return bundle["model"].predict_proba(encoded)[:, 1]

## 7. Four model configurations with household-aware cross-validation

Every fold is split by household groups. MICE is refitted within each fold. SMOTENC is applied only to the training part of a fold and never to its validation part. The threshold is selected from training-only out-of-fold predictions; no held-out test labels are used to choose it.

In [ ]:
STRATEGIES = {
    "SMOTENC_no_ch_weight": {"smote": True, "weights": False},
    "ch_weight_no_SMOTENC": {"smote": False, "weights": True}
}
CONFIGS = [(m, s) for s in STRATEGIES for m in MODEL_PARAMS]

inner_cv = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
oof_probabilities, selected_thresholds, fitted_models = {}, {}, {}
cv_rows = []

for model_name, strategy in CONFIGS:
    key = f"{model_name} | {strategy}"
    oof = np.full(len(X_train), np.nan)
    weighted_objective = STRATEGIES[strategy]["weights"]
    for fold, (tr, va) in enumerate(inner_cv.split(X_train, y_train, groups=g_train), start=1):
        assert set(g_train.iloc[tr]).isdisjoint(set(g_train.iloc[va]))
        bundle = fit_configuration(
            model_name, strategy,
            X_train.iloc[tr], y_train.iloc[tr], w_train.iloc[tr],
            X_train.iloc[va]
        )
        oof[va] = predict_bundle(bundle, X_train.iloc[va])
        print(key, "| fold", fold, "complete")

    if np.isnan(oof).any():
        raise RuntimeError(f"Missing OOF predictions for {key}.")
    oof_probabilities[key] = oof
    threshold_weight = w_train if weighted_objective else None
    precision, recall, thresholds = precision_recall_curve(
        y_train, oof, sample_weight=threshold_weight
    )
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best = int(np.nanargmax(f1_values))
    selected_thresholds[key] = float(thresholds[best]) if len(thresholds) else 0.5
    cv_rows.append({
        "model": model_name, "strategy": strategy,
        "threshold": selected_thresholds[key],
        "training_oof_f1": float(f1_values[best]),
        "training_oof_roc_auc": roc_auc_score(y_train, oof, sample_weight=threshold_weight),
        "training_oof_pr_auc": average_precision_score(y_train, oof, sample_weight=threshold_weight)
    })

cv_results = pd.DataFrame(cv_rows)
display(cv_results.round(4))

## 8. Held-out evaluation: metrics, confusion matrices and ROC-AUC

The test set contains households never used for model fitting, imputation, resampling, threshold selection or cross-validation. Confusion matrices show child counts (unweighted); weighted and unweighted metrics are both reported.

In [ ]:
test_probabilities, test_predictions, heldout_bundles = {}, {}, {}
metric_rows = []
for model_name, strategy in CONFIGS:
    key = f"{model_name} | {strategy}"
    bundle = fit_configuration(
        model_name, strategy, X_train, y_train, w_train, X_test
    )
    probability = predict_bundle(bundle, X_test)
    prediction = (probability >= selected_thresholds[key]).astype(int)
    heldout_bundles[key] = bundle
    test_probabilities[key] = probability
    test_predictions[key] = prediction

    metric_rows.append({
        "model": model_name, "strategy": strategy,
        "threshold": selected_thresholds[key],
        "weighted_precision": precision_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_recall": recall_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_f1": f1_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_roc_auc": roc_auc_score(y_test, probability, sample_weight=w_test),
        "weighted_pr_auc": average_precision_score(y_test, probability, sample_weight=w_test),
        "unweighted_precision": precision_score(y_test, prediction, zero_division=0),
        "unweighted_recall": recall_score(y_test, prediction, zero_division=0),
        "unweighted_f1": f1_score(y_test, prediction, zero_division=0),
        "unweighted_roc_auc": roc_auc_score(y_test, probability),
        "unweighted_pr_auc": average_precision_score(y_test, probability),
        "accuracy": accuracy_score(y_test, prediction)
    })

heldout_results = pd.DataFrame(metric_rows)
display(heldout_results.sort_values("weighted_f1", ascending=False).round(4))

In [ ]:
# Confusion matrices: raw child counts and row-normalized proportions.
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (key, pred) in zip(axes.ravel(), test_predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["No wasting", "Any wasting"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format=",")
    ax.set_title(key)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Observed")
fig.suptitle("Held-out confusion matrices (child counts)", fontsize=14)
plt.tight_layout()
plt.show()

cm_rows = []
for key, pred in test_predictions.items():
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    cm_rows.append({"model_configuration": key, "TN": int(tn), "FP": int(fp),
                    "FN": int(fn), "TP": int(tp)})
confusion_results = pd.DataFrame(cm_rows)
display(confusion_results)

In [ ]:
# ROC curves: solid = unweighted, dashed = weighted by child sample weights.
fig, ax = plt.subplots(figsize=(10, 8))
for key, prob in test_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_u = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, label=f"{key} | unweighted AUC={auc_u:.3f}")
    fpr_w, tpr_w, _ = roc_curve(y_test, prob, sample_weight=w_test)
    auc_w = roc_auc_score(y_test, prob, sample_weight=w_test)
    ax.plot(fpr_w, tpr_w, linestyle="--", label=f"{key} | weighted AUC={auc_w:.3f}")
ax.plot([0, 1], [0, 1], "k:", label="No-discrimination reference")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="Household-held-out ROC curves")
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Explainability: SHAP, built-in importance and permutation importance

SHAP is computed on a sample of the untouched held-out observations. The one-hot feature names are retained so the plots identify the encoded predictor levels. Built-in gain/split importance and permutation importance provide complementary summaries. Permutation importance is calculated on held-out data and can be computationally expensive; reduce `n_repeats` or sample size if needed.

These are predictive explanations, not causal effects.

In [ ]:
# Explain each fitted configuration using the same held-out sample.
SHAP_SAMPLE = min(400, len(X_test))
shap_rng = np.random.RandomState(SEED)
shap_idx = shap_rng.choice(len(X_test), size=SHAP_SAMPLE, replace=False)
explainability_rows = []

for key, bundle in heldout_bundles.items():
    model = bundle["model"]
    encoded = bundle["encoder"].transform(
        bundle["mice_kernel"].impute_new_data(
            new_data=cast_for_mice(X_test.iloc[shap_idx]),
            datasets=0, iterations=5, verbose=False
        ).complete_data(dataset=0)
    )
    feature_names = bundle["feature_names"]
    encoded_dense = encoded.toarray() if hasattr(encoded, "toarray") else np.asarray(encoded)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(encoded_dense)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
    if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    print("\nSHAP summary:", key)
    shap.summary_plot(
        shap_values, encoded_dense, feature_names=feature_names,
        max_display=20, show=True
    )
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    for name, value in zip(feature_names, mean_abs_shap):
        explainability_rows.append({
            "model_configuration": key,
            "feature": name,
            "mean_abs_shap": float(value)
        })

    # Native tree-based feature importance (gain for XGBoost; split/gain for LightGBM).
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
        top_idx = np.argsort(importance)[::-1][:20]
        plt.figure(figsize=(9, 6))
        plt.barh(np.array(feature_names)[top_idx][::-1], np.array(importance)[top_idx][::-1])
        plt.title(f"Built-in feature importance — {key}")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.show()

shap_importance = pd.DataFrame(explainability_rows)
display(
    shap_importance.groupby(["model_configuration", "feature"])["mean_abs_shap"]
    .mean().reset_index()
    .sort_values(["model_configuration", "mean_abs_shap"], ascending=[True, False])
    .groupby("model_configuration").head(15)
)

In [ ]:
# Permutation importance on held-out rows for a complementary model-agnostic ranking.
# This operates on the already imputed/encoded test matrix for each fitted model.
perm_rows = []
for key, bundle in heldout_bundles.items():
    model = bundle["model"]
    Xenc = bundle["encoder"].transform(
        bundle["mice_kernel"].impute_new_data(
            new_data=cast_for_mice(X_test),
            datasets=0, iterations=5, verbose=False
        ).complete_data(dataset=0)
    )
    result = permutation_importance(
        model, Xenc, y_test, scoring="roc_auc",
        n_repeats=5, random_state=SEED, n_jobs=-1
    )
    names = bundle["feature_names"]
    top = np.argsort(result.importances_mean)[::-1][:20]
    for j in top:
        perm_rows.append({
            "model_configuration": key,
            "feature": names[j],
            "permutation_auc_decrease": float(result.importances_mean[j]),
            "permutation_std": float(result.importances_std[j])
        })

permutation_results = pd.DataFrame(perm_rows)
display(permutation_results.sort_values(
    ["model_configuration", "permutation_auc_decrease"],
    ascending=[True, False]
))

## 10. Automated comparison discussion and export

The discussion below is generated from the actual held-out results after the notebook is run. Do not treat it as a result before execution.

In [ ]:
best_wf1 = heldout_results.loc[heldout_results["weighted_f1"].idxmax()]
best_auc = heldout_results.loc[heldout_results["weighted_roc_auc"].idxmax()]
print("RESULTS DISCUSSION")
print("-" * 60)
print(f"Highest weighted F1: {best_wf1['model']} / {best_wf1['strategy']} "
      f"(F1={best_wf1['weighted_f1']:.4f}; threshold={best_wf1['threshold']:.3f}).")
print(f"Highest weighted ROC-AUC: {best_auc['model']} / {best_auc['strategy']} "
      f"(ROC-AUC={best_auc['weighted_roc_auc']:.4f}).")
for _, row in heldout_results.iterrows():
    print(f"- {row['model']} | {row['strategy']}: "
          f"weighted P/R/F1={row['weighted_precision']:.3f}/"
          f"{row['weighted_recall']:.3f}/{row['weighted_f1']:.3f}; "
          f"weighted ROC-AUC={row['weighted_roc_auc']:.3f}; "
          f"unweighted ROC-AUC={row['unweighted_roc_auc']:.3f}.")
print("\nInterpretation guide:")
print("* Higher recall means fewer children with wasting are missed, often at the cost of more false positives.")
print("* Confusion matrices expose false negatives and false positives at the selected training-only threshold.")
print("* ROC-AUC measures ranking across thresholds; PR-AUC is especially informative when the positive class is uncommon.")
print("* Weighted metrics incorporate child sampling weights; unweighted metrics describe this analytic sample.")
print("* Household grouping prevents sibling/household leakage but does not itself account for every MICS survey-design feature.")

OUTPUT = Path("outputs") / "any_wasting_household_aware_mice"
OUTPUT.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(OUTPUT / "grouped_cross_validation.csv", index=False)
heldout_results.to_csv(OUTPUT / "heldout_metrics.csv", index=False)
confusion_results.to_csv(OUTPUT / "heldout_confusion_matrices.csv", index=False)
shap_importance.to_csv(OUTPUT / "shap_feature_importance.csv", index=False)
permutation_results.to_csv(OUTPUT / "permutation_feature_importance.csv", index=False)
print("Saved outputs to:", OUTPUT.resolve())

## 11. Methodological notes and interpretation

1. **Household dependence:** siblings share household conditions and are not independent observations. Grouping by `division` + `hh1` prevents children from the same household crossing train/test or CV boundaries. This is a household-grouped predictive validation design, not a full complex-survey variance estimator.
2. **MICE:** the imputation kernel is fitted only on each training fold and applied to its validation fold. The outer test data are imputed using a kernel fitted on the complete training split. This avoids leakage from held-out outcomes or predictor distributions.
3. **SMOTENC:** applied only to training data. One-hot encoding follows SMOTENC because synthetic interpolation should operate on categorical columns as categories, not as one-hot dummy variables.
4. **Survey weights:** child sample weights are not class weights. The SMOTENC/no-weight strategy deliberately excludes them from model fitting. The no-SMOTE/weighted strategy uses them in model fitting. Weighted and unweighted metrics are both reported for fair interpretation.
5. **Confusion matrix:** TN = correctly identified non-wasting children; FP = non-wasting children flagged as wasting; FN = wasting children missed; TP = correctly identified wasting children.
6. **ROC-AUC and PR-AUC:** ROC-AUC measures discrimination across thresholds. PR-AUC is often more revealing when wasting is a minority outcome.
7. **Explainability:** SHAP describes contributions to model predictions; built-in tree importance summarizes model split/gain usage; permutation importance measures how much held-out ROC-AUC falls when a feature is shuffled. None establishes causation.
8. **No fabricated results:** run all cells from top to bottom using the revised MICS data. Interpret the generated CSVs and plots only after successful execution.

**Reproducibility caution:** MICE, model fitting and SHAP may take substantial time. For faster testing, reduce `CV_FOLDS`, the number of MICE iterations, estimator counts, or `SHAP_SAMPLE`; restore the planned settings for final analysis.